<img src="https://raw.githubusercontent.com/OmarLi-360/excel-corporativo-12-claves/main/assets/logo-drbits.jpg" alt="Dr. Bits" width="170">

# Clave 1 · Estructura antes que fórmulas
**Excel Corporativo · 12 Claves** — Datos tabulares y Tablas de Excel · ⏱ 30 min

[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/OmarLi-360/excel-corporativo-12-claves/blob/main/notebooks/01_estructura_y_tablas.ipynb) &nbsp; [Volver al tablero](https://omarli-360.github.io/excel-corporativo-12-claves/#clave-01)

**Al terminar podrás:**
1. Reconocer en pantalla un reporte que no se puede analizar.
2. Ver con un gráfico cuánto se equivoca una suma hecha sobre un reporte.
3. Entender qué cambia cuando la misma información se ordena en una tabla.
4. Elegir un gráfico que solo es posible con datos ordenados.

---
### Cómo usar este cuaderno (no necesitas programar)
1. **Guarda tu copia:** *Archivo › Guardar una copia en Drive*. Así tus cambios no se pierden.
2. **Ejecuta todo:** *Entorno de ejecución › Ejecutar todas* (Ctrl+F9). Si Colab avisa que el cuaderno no es de Google, elige *Ejecutar de todos modos*.
3. **Juega con los controles:** las celdas con listas desplegables a la derecha se pueden cambiar. Elige una opción y pulsa el botón ▶ de esa celda para redibujar.

El código está oculto a propósito: aquí importa entender el dato y el gráfico. Si tienes curiosidad, haz doble clic en una celda para verlo.

In [ ]:
# @title Preparar el entorno (se ejecuta sola con «Ejecutar todas»)
import os, sys, warnings
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
from IPython.display import display, Markdown, HTML
warnings.filterwarnings("ignore")

EN_COLAB = "google.colab" in sys.modules
REPO = "OmarLi-360/excel-corporativo-12-claves"
BASE_URL = f"https://raw.githubusercontent.com/{REPO}/main/data/"
BASE_LOCAL = "../data/"

def ruta(nombre):
    local = os.path.join(BASE_LOCAL, nombre)
    return local if os.path.exists(local) else BASE_URL + nombre

# --- formatos colombianos -------------------------------------------------
def miles(x):    return f"{x:,.0f}".replace(",", ".")
def pesos(x):    return "$ " + miles(x)
def millones(x): return "$ " + f"{x/1e6:,.0f}".replace(",", ".") + " M"
def porc(v, signo=False):
    t = f"{v:+.1%}" if signo else f"{v:.1%}"
    return t.replace(".", ",").replace("%", " %")
MESES = ["Ene", "Feb", "Mar", "Abr", "May", "Jun", "Jul", "Ago", "Sep", "Oct", "Nov", "Dic"]
MM = "Millones de pesos"

# --- plantilla de gráficos Dr. Bits ----------------------------------------
AZUL, TEAL, MAGENTA, GRIS = "#4A5AD8", "#0E9BB8", "#D0336F", "#9AA3BF"
BIEN, MAL = "#15784A", "#B42318"
ejes = dict(gridcolor="#E4E8F2", zeroline=False, linecolor="#C9D0E2", exponentformat="none", separatethousands=True)
pio.templates["drbits"] = go.layout.Template(layout=dict(
    font=dict(family="Source Sans 3, Segoe UI, Arial, sans-serif", size=14, color="#11153A"),
    title=dict(font=dict(size=18, color="#00006A"), x=0, xanchor="left"),
    colorway=[AZUL, TEAL, MAGENTA], plot_bgcolor="white", paper_bgcolor="white",
    xaxis=ejes, yaxis=ejes, separators=",.", hoverlabel=dict(font_size=13),
    margin=dict(t=80, l=10, r=30, b=40)))
pio.templates.default = "plotly_white+drbits"

def eje_meses(fig, fechas, cada=None):
    """Etiquetas de meses en español en ejes de tiempo (Plotly las pone en inglés)."""
    f = pd.to_datetime(pd.Series(sorted(set(pd.to_datetime(list(fechas))))))
    paso = cada or (1 if len(f) <= 13 else 3)
    f = f.iloc[::paso]
    fig.update_xaxes(tickvals=list(f), ticktext=[f"{MESES[d.month - 1]} {d.year}" for d in f])

def dar_aire(fig, maximo, eje="x"):
    (fig.update_xaxes if eje == "x" else fig.update_yaxes)(range=[0, maximo * 1.22])

# --- revisor de retos -----------------------------------------------------
def revisar(respuesta, esperado, explicacion=""):
    if respuesta == "Elige una opción":
        display(Markdown("**Elige una opción en la lista de la derecha** y vuelve a ejecutar esta celda con ▶."))
    elif str(respuesta) == str(esperado):
        display(Markdown(f"✅ **Correcto.** {explicacion}"))
    else:
        display(Markdown(f"❌ **Todavía no.** Revisa el gráfico de arriba e inténtalo otra vez. *(Elegiste: {respuesta})*"))

# --- datos de Distribuidora Andina ----------------------------------------
ventas = pd.read_csv(ruta("ventas.csv"), parse_dates=["fecha"])
clientes = pd.read_csv(ruta("clientes.csv"))
productos = pd.read_csv(ruta("productos.csv"))
vendedores = pd.read_csv(ruta("vendedores.csv"))
sedes = pd.read_csv(ruta("sedes.csv"))
metas = pd.read_csv(ruta("metas.csv"))
df = (ventas
      .merge(sedes[["id_sede", "ciudad"]], on="id_sede", validate="many_to_one")
      .merge(productos[["sku", "producto", "categoria"]], on="sku", validate="many_to_one")
      .merge(clientes[["id_cliente", "razon_social", "segmento"]], on="id_cliente", validate="many_to_one")
      .merge(vendedores[["id_vendedor", "nombre_vendedor"]], on="id_vendedor", validate="many_to_one")
      .assign(anio=lambda d: d.fecha.dt.year, mes=lambda d: d.fecha.dt.month))
validas = df[df["estado"] != "Anulada"]          # las anuladas no son venta
ORDEN_CIUDADES = (validas[validas.anio == 2025].groupby("ciudad")["subtotal"].sum()
                  .sort_values(ascending=False).index.tolist())
display(Markdown(f"Datos listos: **{miles(len(ventas))} líneas de venta**, {len(clientes)} clientes, {len(productos)} productos. "
                 f"Fuente: {'archivos locales' if os.path.exists(BASE_LOCAL) else 'GitHub'}."))

## 1 · El reporte que llega por correo
El libro de práctica tiene la hoja **Reporte_Mal_Estructurado**: el informe de ventas de 2025 armado para imprimir. Así lo ve Python: cada celda vacía (por una celda combinada, una fila en blanco o un título) aparece en blanco.

In [ ]:
# @title Ver el reporte tal como llega
reporte = pd.read_excel(ruta("Distribuidora_Andina_practica.xlsx"), sheet_name="Reporte_Mal_Estructurado", header=None)
fila_enc = reporte.index[reporte.eq("Ciudad").any(axis=1)][0]
col = reporte.columns[reporte.loc[fila_enc].eq("Ciudad")][0]
vista = reporte.iloc[fila_enc - 2:fila_enc + 16, col:col + 6].fillna("")
vista.columns = list("BCDEFG")[:vista.shape[1]]
display(vista.style.hide(axis="index").set_properties(**{"font-size": "12px"}))

Fíjate en tres cosas: la ciudad aparece una sola vez por bloque (era una **celda combinada**), hay filas **Subtotal** mezcladas con los registros y algunas cifras están escritas como **texto** (por ejemplo `1.204,9 M`).

## 2 · ¿Cuánto se equivoca una suma rápida?
Alguien con prisa selecciona todas las cifras y usa SUMA. Excel ignora los textos y suma subtotales y total como si fueran ventas.

In [ ]:
# @title Preparar el cálculo (Colab hace los pasos por ti)
datos_rep = reporte.iloc[fila_enc + 2:, col:col + 6].copy()
datos_rep.columns = ["ciudad", "linea", "T1", "T2", "T3", "T4"]
es_sub = datos_rep["linea"].astype(str).str.startswith(("Subtotal", "TOTAL"))
es_vacia = datos_rep["linea"].isna()
valores = datos_rep[["T1", "T2", "T3", "T4"]]
es_texto = valores.map(lambda v: isinstance(v, str))
def a_numero(v):
    return float(v.replace(" M", "").replace(".", "").replace(",", ".")) * 1e6 if isinstance(v, str) else v
suma_rapida = valores[~es_texto].apply(pd.to_numeric, errors="coerce").sum().sum()
correcta = valores[~es_sub & ~es_vacia].map(a_numero).astype(float).sum().sum()
veces = suma_rapida / correcta
display(Markdown(f"Filas de subtotal o total: **{es_sub.sum()}** · filas vacías o notas: **{es_vacia.sum()}** · cifras escritas como texto: **{int(es_texto.values.sum())}**"))

> **Ficha de decisión**
>
> | | |
> |---|---|
> | **Pregunta** | ¿Cuánto infla el total una suma hecha sobre el reporte? |
> | **Qué se grafica** | Dos barras: la suma rápida y la suma correcta, en millones |
> | **Por qué esta forma** | Solo hay dos cifras y el mensaje es la diferencia: dos barras lo muestran sin explicación. |
> | **Qué se descartó** | Una tarjeta con el número: se pierde la comparación visual. |
> | **Cómo se lee** | La barra magenta mide la suma rápida y la azul la correcta. La diferencia es el error. |
> | **En Excel 365** | Barra agrupada con dos celdas de datos. |

In [ ]:
# @title Gráfico 1 · Dos totales para la misma hoja
comp = pd.DataFrame({"cálculo": ["Seleccionar todo y SUMA", "Solo registros, texto convertido"], "millones": [suma_rapida / 1e6, correcta / 1e6]})
fig = px.bar(comp, x="millones", y="cálculo", orientation="h", color="cálculo", color_discrete_sequence=[MAGENTA, AZUL],
             text=comp["millones"].map(lambda v: f"$ {v:,.0f} M".replace(",", ".")),
             title=f"La suma rápida infla el total {veces:.1f} veces".replace(".", ","))
fig.update_traces(textposition="outside", cliponaxis=False)
fig.update_layout(showlegend=False, xaxis_title=MM, yaxis_title="", height=280)
dar_aire(fig, suma_rapida / 1e6)
fig.show()

## 3 · La misma información, ordenada
Colab hizo cuatro pasos, los mismos que harías en Excel: **repetir la ciudad** en cada fila (en Excel: separar celdas y rellenar), **quitar subtotales y filas vacías** (filtrar y eliminar), **convertir el texto en número** y **llevar los trimestres a una columna** (en Power Query: anular dinamización de columnas).

In [ ]:
# @title Ver el antes y el después
ordenada = datos_rep.copy()
ordenada["ciudad"] = ordenada["ciudad"].ffill()
ordenada = ordenada[~es_sub & ~es_vacia]
for t in ["T1", "T2", "T3", "T4"]:
    ordenada[t] = ordenada[t].map(a_numero).astype(float)
ordenada = ordenada.melt(id_vars=["ciudad", "linea"], var_name="trimestre", value_name="subtotal")
antes = pd.DataFrame({"": ["Filas", "Columnas", "Celdas vacías dentro de los datos", "Subtotales", "Cifras como texto"],
                      "Reporte": [len(datos_rep), 6, int(datos_rep[["ciudad"]].isna().sum().iloc[0]), int(es_sub.sum()), int(es_texto.values.sum())],
                      "Tabla ordenada": [len(ordenada), 4, 0, 0, 0]})
display(antes.style.hide(axis="index"))
display(ordenada.head(8).style.format({"subtotal": pesos}).hide(axis="index"))

Tiene más filas y ninguna ambigüedad. Ahora sí se puede graficar cualquier combinación.

> **Ficha de decisión**
>
> | | |
> |---|---|
> | **Pregunta** | ¿Cómo evolucionó cada línea durante 2025 en la ciudad que elijas? |
> | **Qué se grafica** | Eje X: trimestre · Eje Y: venta (millones) · Color: línea de producto · Filtro: ciudad |
> | **Por qué esta forma** | Los trimestres son tiempo: las líneas muestran la tendencia de cada producto. |
> | **Qué se descartó** | Columnas apiladas: no permiten comparar la tendencia de cada línea. |
> | **Cómo se lee** | Sigue cada línea de izquierda a derecha. El título dice cuál creció más de T1 a T4. |
> | **En Excel 365** | Tabla dinámica con trimestre en Filas y línea en Columnas › Gráfico de líneas. Con el reporte original era imposible. |

In [ ]:
# @title Gráfico 2 · Evolución por trimestre (elige la ciudad)
ciudad = "Todas"  # @param ["Todas", "Bogotá", "Medellín", "Cali", "Barranquilla", "Bucaramanga", "Pereira"]
base = ordenada if ciudad == "Todas" else ordenada[ordenada.ciudad == ciudad]
serie = base.groupby(["trimestre", "linea"], as_index=False)["subtotal"].sum()
serie["millones"] = serie["subtotal"] / 1e6
var = serie.pivot(index="linea", columns="trimestre", values="subtotal")
var = (var["T4"] / var["T1"] - 1).sort_values()
fig = px.line(serie, x="trimestre", y="millones", color="linea", markers=True,
              color_discrete_sequence=[AZUL, TEAL, MAGENTA, "#7A5AC8", "#3E8E5E"],
              title=f"{ciudad if ciudad != 'Todas' else 'Toda la empresa'}: {var.index[-1]} fue la línea que más creció de T1 a T4 ({porc(var.iloc[-1], signo=True)})")
fig.update_layout(yaxis_title=MM, xaxis_title="", legend_title="")
fig.show()

## 4 · La calidad también es de significado
La tabla de ventas pasa todas las pruebas de forma (sin vacíos ni duplicados), pero contiene filas que **no son venta**: las facturas anuladas. Ninguna prueba automática lo detecta; lo detecta quien conoce el negocio.

> **Ficha de decisión**
>
> | | |
> |---|---|
> | **Pregunta** | ¿Cuánto dinero hay en facturas anuladas? |
> | **Qué se grafica** | Barras: venta por estado de la factura, 2024–2025 |
> | **Por qué esta forma** | Pocas categorías comparadas en una medida. |
> | **Qué se descartó** | Torta: la parte anulada es tan pequeña que no se vería. |
> | **Cómo se lee** | La barra Anulada parece pequeña, pero son cientos de millones que inflarían cualquier informe. |
> | **En Excel 365** | Tabla dinámica con estado en Filas y subtotal en Valores. |

In [ ]:
# @title Gráfico 3 · Venta por estado
est = df.groupby("estado")["subtotal"].sum().reindex(["Pagada", "Pendiente", "Anulada"]) / 1e6
fig = go.Figure(go.Bar(x=est.index, y=est.values, marker_color=[AZUL, TEAL, MAGENTA],
                       text=[f"$ {v:,.0f} M".replace(",", ".") for v in est.values], textposition="outside", cliponaxis=False))
fig.update_layout(title=f"Las anuladas suman {millones(est['Anulada'] * 1e6)}: si no se filtran, se cuentan como venta",
                  yaxis_title=MM, height=380)
dar_aire(fig, est.max(), "y")
fig.show()

## 5 · Retos de interpretación

In [ ]:
# @title Reto 1 · ¿Cuántas veces infla el total la suma rápida?
respuesta = "Elige una opción"  # @param ["Elige una opción", "1,5 veces", "2 veces", "3 veces", "4 veces"]
revisar(respuesta, f"{round(veces)} veces", """La suma rápida cuenta cada venta dos veces (registro y subtotal) y además el total general.""")

In [ ]:
# @title Reto 2 · Para comparar trimestres de cada línea, ¿qué estructura necesitas?
respuesta = "Elige una opción"  # @param ["Elige una opción", "Una columna por trimestre", "Una columna trimestre y una columna subtotal", "Una hoja por trimestre"]
revisar(respuesta, "Una columna trimestre y una columna subtotal", """El trimestre es una variable: va en una columna propia.""")

In [ ]:
# @title Reto 3 · ¿Qué gráfico usarías para mostrar cómo evolucionó Bogotá por trimestre?
respuesta = "Elige una opción"  # @param ["Elige una opción", "Torta", "Líneas", "Dispersión", "Mapa de árbol"]
revisar(respuesta, "Líneas", """El eje horizontal es tiempo: las líneas muestran la tendencia.""")

## Apóyate en IA
Si recibes un reporte con otra forma, pídele a una IA que te diga cómo ordenarlo en Excel:

```text
Tengo un reporte de Excel con este encabezado y estas 8 filas de muestra (datos anonimizados):
[pega aquí]
Dime qué problemas de estructura tiene para hacer una tabla dinámica y los pasos en Excel 365 en español
para convertirlo en una tabla con una fila por registro. No inventes datos.
```

> **Verifica siempre.** Compara la respuesta de la IA con un dato que ya conozcas. No pegues datos personales reales (Ley 1581 de 2012): estos datos son ficticios, los de tu empresa no.

## Resumen
* Un reporte sirve para leer; una tabla ordenada sirve para analizar.
* Una suma sobre un reporte cuenta subtotales e ignora textos: el error es silencioso.
* Los gráficos de evolución y comparación solo son posibles con datos ordenados.

**Siguiente:** [Clave 5 · Buscar y cruzar sin #N/D](https://omarli-360.github.io/excel-corporativo-12-claves/#clave-05) · [Cuaderno 05](https://colab.research.google.com/github/OmarLi-360/excel-corporativo-12-claves/blob/main/notebooks/05_busqueda_y_cruce.ipynb)

<sub>Dr. Bits · ADNBITS S.A.S. · Contenido CC BY-SA 4.0 · Datos ficticios de Distribuidora Andina S.A.S.</sub>